
<table align="center" style="margin: 0 auto; border: 0; width: auto;">
<tr><td align="center" style="border: 0; padding: 0; text-align: center;">
<img src="https://logoyar.com/content/wp-content/uploads/2021/04/sharif-university-logo.png" width="155" alt="Sharif University of Technology logo" style="background: white; padding: 8px; border-radius: 50%;" />
</td></tr></table>

<h2 align="center">SHARIF UNIVERSITY OF TECHNOLOGY</h2>

<h3 align="center">Department of Electrical Engineering</h3>


<h1 align="center">Deep Generative Models</h1>

<h2 align="center">Practical Assignment 2 | Notebook 1 | Variational Autoencoders</h2>






<table align="center" style="margin: 15px auto 25px auto; width: 65%; border-collapse: collapse; text-align: center;">
  <tr>
    <th width="50%" style="text-align: center; padding: 12px; border-bottom: 1px solid;">
      Student Name
    </th>
    <th width="50%" style="text-align: center; padding: 12px; border-bottom: 1px solid;">
      Student ID
    </th>
  </tr>
  <tr>
    <td style="text-align: center; padding: 20px 12px;">
      ______________________
    </td>
    <td style="text-align: center; padding: 20px 12px;">
      ______________________
    </td>
  </tr>
</table>


---

### **Assignment Overview & Instructions:**

<table align="center" style="margin-left: auto; margin-right: auto; border-collapse: collapse; width: auto;">
<thead>
<tr>
<th style="text-align: center; padding: 8px 14px;">Part</th>
<th style="text-align: left; padding: 8px 14px;">Topic</th>
<th style="text-align: center; padding: 8px 14px;">Points</th>
</tr>
</thead>
<tbody>
<tr>
<td align="center"><b>I</b></td>
<td>VAE Generation and Latent Space Exploration</td>
<td align="center"><b>40</b></td>
</tr>
<tr>
<td align="center"><b>II</b></td>
<td>Gaussian and Mixture Priors</td>
<td align="center"><b>15</b></td>
</tr>
<tr>
<td align="center"><b>III</b></td>
<td>Denoising Variational Autoencoders</td>
<td align="center"><b>45</b></td>
</tr>
<tr>
<td></td>
<td><b>Total</b></td>
<td align="center"><b>100</b></td>
</tr>
</tbody>
</table>


In this notebook, you will train VAEs for image generation, compare different priors, and restore images with missing pixels.


Complete the tasks in order. Run the checks beside each implementation before starting the related training. Include figures and short written answers where requested. *Quick-test outputs are for debugging only.*

Fashion-MNIST and MNIST are downloaded automatically on the first run and cached for subsequent runs. Parts I and III use PyTorch `DataLoader`; Part II uses a synthetic distribution. Training runs on CUDA when available and otherwise on CPU (more slowly).


**Environment:** Python 3.10+, PyTorch 2.3+, NumPy, pandas, Matplotlib, tqdm, Pillow >= 9.1. An Internet connection is needed for the first dataset download unless the files are already cached. Checkpoints and figures are stored locally. On Kaggle, download the checkpoint before the session expires.




---

# **Part I — Training a VAE (40 points)**

**Goal.** Train an unconditional VAE and inspect its generation and latent-space behavior.

- **Data:** 28×28 Fashion-MNIST, 54,000 training and 6,000 validation images; the official 10,000-image test split is held out. Class labels are not used for training.
- **Work:** reparameterization, VAE forward pass, Gaussian KL, ELBO, validation, an optimizer step, independent prior sampling, and interpolation.
- **Evidence:** training progression (fixed latent vectors), generated samples, test reconstructions, matched pixel/latent interpolations, and BCE/KL curves.

The model is trained with the standard $\beta=1$ objective:

$$
\mathcal{L}_{\mathrm{VAE}}(x)
=
\underbrace{\mathbb{E}_{q_\phi(z\mid x)}[-\log p_\theta(x\mid z)]}_{\mathrm{reconstruction}}
+
\underbrace{D_{\mathrm{KL}}(q_\phi(z\mid x)\Vert p(z))}_{\mathrm{regularization}},
\qquad p(z)=\mathcal{N}(0,I).
$$

For binary-valued pixels modeled by a Bernoulli likelihood, the first term is binary cross-entropy. Our display images use pixel probabilities, not randomized Bernoulli samples.


In [ ]:
import math
import os, io, gzip, struct, random, time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

SEED = 2026
EPOCHS = 60            # standard training budget; CPU runs will be slower
BATCH_SIZE = 256
LATENT_DIM = 32
BETA = 1.0            # standard VAE objective; don't silently alter beta for nicer pictures
LR = 1e-3
MAX_TRAIN = 60_000     # full Fashion-MNIST train split
NUM_WORKERS = 0       # all 28x28 images are preloaded into RAM, no ZIP/JPEG bottleneck
OUT = Path('/kaggle/working/vae_homework_2026_fashion') if Path('/kaggle/working').exists() else Path('./vae_homework_2026_fashion')
OUT.mkdir(parents=True, exist_ok=True)
CKPT = OUT / 'fashion_vae_checkpoint.pt'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if USE_AMP:
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
print('device:', DEVICE, '| GPU:', torch.cuda.get_device_name(0) if USE_AMP else 'none')
print('output folder:', OUT)

### I.1 — Download and load the dataset

The provided loader downloads the four original IDX archives over HTTPS on the first run, checks their MD5 digests, and reuses a local cache thereafter. It also supports an explicitly supplied local folder via the `FASHION_MNIST_DATA_DIR` environment variable and a narrowly scoped Kaggle attachment. There is no scan of unrelated datasets.

**Output:** source location and tensor shapes. The DataLoaders are provided because loader implementation is not assessed in this assignment.

In [ ]:
def read_idx_images(path):
    opener = gzip.open if str(path).endswith('.gz') else open
    with opener(path, 'rb') as f:
        header = f.read(16)
        magic, count, height, width = struct.unpack('>IIII', header)
        if magic != 2051 or (height, width) != (28, 28):
            raise ValueError(f'Unexpected IDX image file: {path}, header={(magic,count,height,width)}')
        arr = np.frombuffer(f.read(), dtype=np.uint8).copy()
        return arr.reshape(count, 28, 28)

def read_idx_labels(path):
    opener = gzip.open if str(path).endswith('.gz') else open
    with opener(path,'rb') as f:
        magic, count = struct.unpack('>II', f.read(8))
        if magic != 2049: raise ValueError('Invalid IDX labels file')
        return np.frombuffer(f.read(), dtype=np.uint8).copy().reshape(count)

def find_by_prefix(root, filename_prefix):
    # Search only inside the Fashion-MNIST dataset folder, never all Kaggle inputs.
    root = Path(root)
    for pattern in (filename_prefix + '*', '*/' + filename_prefix + '*', '*/*/' + filename_prefix + '*'):
        for p in root.glob(pattern):
            if p.is_file():
                return p
    return None

def load_root(root):
    root = Path(root)
    train_idx = find_by_prefix(root,'train-images-idx3-ubyte')
    test_idx = find_by_prefix(root,'t10k-images-idx3-ubyte')
    if train_idx and test_idx:
        tr=read_idx_images(train_idx); te=read_idx_images(test_idx)
        label_file=find_by_prefix(root,'t10k-labels-idx1-ubyte')
        labs=read_idx_labels(label_file) if label_file else None
        return tr,te,labs, f'IDX files in {root}'
    tr_csv = find_by_prefix(root,'fashion-mnist_train.csv')
    te_csv = find_by_prefix(root,'fashion-mnist_test.csv')
    if tr_csv and te_csv:
        tr = pd.read_csv(tr_csv, dtype=np.uint8).to_numpy()
        te = pd.read_csv(te_csv, dtype=np.uint8).to_numpy()
        return tr[:,1:].reshape(-1,28,28).copy(), te[:,1:].reshape(-1,28,28).copy(),te[:,0].copy(),f'CSV files in {root}'
    return None

# The same official checksums published by Zalando and used in torchvision.
import hashlib
import urllib.request

CHECKSUMS = {
    'train-images-idx3-ubyte.gz': '8d4fb7e6c68d591d4c3dfef9ec88bf0d',
    'train-labels-idx1-ubyte.gz': '25c81989df183df01b3e8a0aad5dffbe',
    't10k-images-idx3-ubyte.gz': 'bef4ecab320f06d8554ea6380940ec79',
    't10k-labels-idx1-ubyte.gz': 'bb300cfdad3c16e7a12a480ee83cd310',
}

# An explicit folder is supported for teaching labs, but is NOT needed by default.
DATA_CACHE = Path(os.environ.get('FASHION_MNIST_CACHE',
                  str(Path.home() / '.cache' / 'fashion_mnist_vae'))).expanduser()
MIRRORS = [
    'https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/',
    'https://storage.googleapis.com/tensorflow/tf-keras-datasets/',
]

def md5_file(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

def fetch_verified_file(filename, expected_hash, cache_dir=DATA_CACHE):
    """Download once via HTTPS; reuse cache; verify content; never retain partial files."""
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    destination = cache_dir / filename
    if destination.is_file() and md5_file(destination) == expected_hash:
        print('Cached:', filename)
        return destination
    if destination.exists():
        print('Invalid cached file; downloading again:', filename)
        destination.unlink()

    errors = []
    for base in MIRRORS:
        url = base + filename
        temp_path = destination.with_name(filename + '.part')
        try:
            print('Downloading:', url, flush=True)
            request = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (educational VAE notebook)'})
            with urllib.request.urlopen(request, timeout=25) as source, open(temp_path, 'wb') as out:
                while True:
                    chunk = source.read(1024 * 1024)
                    if not chunk:
                        break
                    out.write(chunk)
            received = md5_file(temp_path)
            if received != expected_hash:
                raise ValueError(f'MD5 mismatch for {filename}: got {received}')
            temp_path.replace(destination)
            return destination
        except Exception as exc:
            errors.append(f'{url} -> {type(exc).__name__}: {str(exc)[:140]}')
            temp_path.unlink(missing_ok=True)

    raise RuntimeError('Could not download official Fashion-MNIST file ' + filename +
                       '.\nIf running on Kaggle, enable Internet in Notebook Settings on the first run.\n' +
                       '\n'.join(errors))


def load_fashion():
    # Reuse a student-provided local folder, if explicitly specified.
    override = os.environ.get('FASHION_MNIST_DATA_DIR')
    if override:
        local = load_root(Path(override).expanduser())
        if local is None:
            raise FileNotFoundError('FASHION_MNIST_DATA_DIR is set but the folder has no valid Fashion-MNIST files: ' + override)
        return local

    # Cache first. This makes a second run fast and offline.
    if all((DATA_CACHE / name).is_file() and
           md5_file(DATA_CACHE / name) == digest for name, digest in CHECKSUMS.items()):
        result = load_root(DATA_CACHE)
        if result is not None:
            return result

    # Optional, Kaggle only: restrict search to folders with matching names.
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        for folder in kaggle_input.iterdir():
            if folder.is_dir() and ('fashion' in folder.name.lower() or 'zalando' in folder.name.lower()):
                local = load_root(folder)
                if local is not None:
                    print('Using attached Kaggle dataset; no download needed.')
                    return local

    # Normal route on a laptop, Kaggle, or Colab: fetch official files automatically.
    for name, digest in CHECKSUMS.items():
        fetch_verified_file(name, digest)
    result = load_root(DATA_CACHE)
    if result is None:
        raise RuntimeError('Fashion-MNIST download finished but its files could not be parsed.')
    return result


train_np, test_np, test_labels, DATA_SOURCE = load_fashion()
assert train_np.shape == (60000, 28, 28) and test_np.shape == (10000, 28, 28)
assert test_labels is not None and len(test_labels) == 10000, 'Test labels needed only to name interpolation endpoints.'
indices = np.random.default_rng(SEED).permutation(60000)
train_idx, val_idx = indices[:54000], indices[54000:]
train_tensor = torch.from_numpy(np.ascontiguousarray(train_np[train_idx])).unsqueeze(1)
val_tensor = torch.from_numpy(np.ascontiguousarray(train_np[val_idx])).unsqueeze(1)
test_tensor = torch.from_numpy(np.ascontiguousarray(test_np)).unsqueeze(1)
train_loader = DataLoader(TensorDataset(train_tensor), batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=USE_AMP)
val_loader = DataLoader(TensorDataset(val_tensor), batch_size=512, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=USE_AMP)
test_loader = DataLoader(TensorDataset(test_tensor), batch_size=512, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=USE_AMP)
print('Data source:', DATA_SOURCE)
print('Train / validation / test:', len(train_tensor), len(val_tensor), len(test_tensor))
print('Label input to model: none')

In [ ]:
fig,axes=plt.subplots(3,10,figsize=(12,4))
for ax,img in zip(axes.flat, train_np[:30]):
    ax.imshow(img,cmap='gray',vmin=0,vmax=255);ax.axis('off')
fig.suptitle('Real training images (not generated)');plt.tight_layout();plt.show()

### I.2 — Reparameterization and KL (8 points)

Implement the pathwise sample and the analytic diagonal-Gaussian KL:

$$z=\mu+\exp(\tfrac12\log\sigma^2)\odot\varepsilon,\quad \varepsilon\sim\mathcal N(0,I),$$
$$D_{\mathrm{KL}}(q\Vert p)=\tfrac12\sum_{j=1}^d (\mu_j^2+\sigma_j^2-1-\log\sigma_j^2).$$

**Interfaces:** `reparameterize(mu, logvar) -> z` (same `[B,D]` shape, differentiable); `gaussian_kl(mu, logvar) -> scalar` (sum latent dimensions, mean batch). The checks below cover shapes, gradients, and the zero-KL case.


In [ ]:
def reparameterize(mu, logvar):
    """Return a differentiable latent sample with the same [B,D] shape."""
    # TODO I.2a (4 points): reparameterization trick
    raise NotImplementedError('TODO I.2a')


def gaussian_kl(mu, logvar):
    """Return mean KL per image, summed over latent coordinates (scalar)."""
    # TODO I.2b (4 points): analytical KL against N(0,I)
    raise NotImplementedError('TODO I.2b')


In [ ]:
# Unit tests: correct formula, shapes, and gradients.
a = torch.zeros(6, 4, requires_grad=True)
b = torch.zeros(6, 4, requires_grad=True)
z = reparameterize(a, b)
assert z.shape == (6, 4) and z.requires_grad
assert torch.allclose(gaussian_kl(a, b), torch.zeros(()), atol=1e-7)
loss_check = z.square().mean() + gaussian_kl(a, b)
loss_check.backward()
assert a.grad is not None and b.grad is not None
assert torch.isfinite(a.grad).all() and torch.isfinite(b.grad).all()
print('PASS — I.2: reparameterization and Gaussian KL')

### I.3 — Model forward pass (4 points)

The convolutional encoder and decoder layers are supplied. Complete `encode`, `decode`, and `forward` to connect them: `forward(x) -> (logits, mu, logvar)`, where logits have shape `[B,1,28,28]` and posterior parameters have shape `[B,LATENT_DIM]`. Sample \(z\) using your `reparameterize` function. Do not apply `sigmoid` to reconstruction logits during training.


In [ ]:
class ConvVAE(nn.Module):
    def __init__(self, latent_dim=32):
        super().__init__()
        self.enc=nn.Sequential(
            nn.Conv2d(1,32,4,2,1),nn.ReLU(inplace=True),  # 14x14
            nn.Conv2d(32,64,4,2,1),nn.ReLU(inplace=True), # 7x7
            nn.Flatten(),
            nn.Linear(64*7*7,256),nn.ReLU(inplace=True)
        )
        self.mu=nn.Linear(256,latent_dim)
        self.logvar=nn.Linear(256,latent_dim)
        self.dec=nn.Sequential(
            nn.Linear(latent_dim,256),nn.ReLU(inplace=True),
            nn.Linear(256,64*7*7),nn.ReLU(inplace=True),
            nn.Unflatten(1,(64,7,7)),
            nn.ConvTranspose2d(64,32,4,2,1),nn.ReLU(inplace=True), # 14x14
            nn.ConvTranspose2d(32,1,4,2,1)                      # 28x28 logits
        )
    def encode(self, x):
        """Return (mu, logvar), each [B, LATENT_DIM]."""
        # TODO I.3 (4 points): connect the provided encoder to posterior heads.
        raise NotImplementedError('TODO I.3: encode')

    def reparameterize(self,mu,logvar):
        return reparameterize(mu, logvar)
    def decode(self, z):
        """Return logits [B,1,28,28]."""
        # TODO I.3: connect the supplied decoder.
        raise NotImplementedError('TODO I.3: decode')

    def forward(self, x):
        """Return (logits, mu, logvar) for a reparameterized latent sample."""
        # TODO I.3: combine encode, reparameterize, and decode.
        raise NotImplementedError('TODO I.3: forward')

In [ ]:
# Test the VAE interface before defining the loss.
_m=ConvVAE(LATENT_DIM).to(DEVICE)
_v=torch.zeros(3,1,28,28,device=DEVICE)
_l,_mu,_lv=_m(_v)
assert _l.shape==_v.shape and _mu.shape==_lv.shape==(3,LATENT_DIM)
assert torch.isfinite(_l).all() and torch.isfinite(_mu).all()
print('PASS — model forward interface')
del _m,_v,_l,_mu,_lv

### I.4 — VAE loss (7 points)

Implement `loss_parts(logits, x, mu, logvar) -> (loss, bce, kl)`. All outputs are scalar tensors in **nats per image**. Use `binary_cross_entropy_with_logits`, summing pixels before averaging the batch; use your KL implementation. Here \(\beta=1\). Keep reconstruction logits untransformed inside the loss.


In [ ]:
def loss_parts(logits, x, mu, lv):
    """Return (objective, BCE, KL), three scalar nats/image tensors."""
    # TODO I.4 (7 points): BCE with logits + beta-weighted KL
    raise NotImplementedError('TODO I.4')

model=ConvVAE(LATENT_DIM).to(DEVICE)
optimizer=torch.optim.Adam(model.parameters(),lr=LR)
scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP,init_scale=128.0)
print('Parameters:',sum(p.numel() for p in model.parameters()))
# Shape and gradient sanity check before full training
x=torch.zeros(4,1,28,28,device=DEVICE)
logits,mu,lv=model(x)
assert logits.shape == x.shape and mu.shape == (4,LATENT_DIM)
assert torch.isfinite(loss_parts(logits,x,mu,lv)[0])
print('Shape/loss sanity check PASSED')
# Objective check: BCE and KL must have the same per-image reduction as in the specification.
with torch.no_grad():
    fake_logits=torch.zeros(2,1,28,28,device=DEVICE)
    fake_img=torch.zeros_like(fake_logits)
    zero_latent=torch.zeros(2,LATENT_DIM,device=DEVICE)
    objective,bce,kl=loss_parts(fake_logits,fake_img,zero_latent,zero_latent)
    expected_bce=28*28*math.log(2)
    assert abs(float(bce)-expected_bce)<.05 and abs(float(kl))<1e-6
    assert abs(float(objective-bce))<1e-6
print('PASS — loss scaling (nats per image)')


### I.5 — Validation and checkpoints (3 points)

Implement `evaluate(loader) -> (bce, kl)` by averaging the per-image losses over the **whole** DataLoader, without gradient updates. The output should be two finite Python floats. The official test set is evaluated only after training; checkpoints are based on the training/validation workflow.

A checkpoint is saved after each epoch. Preview samples use the **same fixed latent vectors** throughout training; differences across preview rows reflect parameter updates rather than new random draws.


In [ ]:
@torch.no_grad()
def decode_probs(z):
    model.eval()
    return torch.sigmoid(model.decode(z.to(DEVICE))).cpu()

def image_grid(images, ncol=8, title='', save_as=None):
    images = images.detach().cpu().float().clamp(0,1)
    n = len(images); nrow = int(np.ceil(n/ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(ncol*1.1, nrow*1.2), squeeze=False)
    for ax in axes.flat: ax.axis('off')
    for ax, image in zip(axes.flat, images):
        ax.imshow(image.squeeze(0), cmap='gray', vmin=0, vmax=1, interpolation='nearest')
    fig.suptitle(title, fontsize=11)
    fig.tight_layout(rect=(0,0,1,.96))
    if save_as: fig.savefig(OUT/save_as, dpi=150, bbox_inches='tight')
    plt.show(); plt.close(fig)

fixed_z=torch.randn(64,LATENT_DIM,generator=torch.Generator().manual_seed(31415))
PREVIEW_EPOCHS = sorted(set([1,3,5,10,20,30,45,EPOCHS]))
previews={}

@torch.no_grad()
def evaluate(loader):
    """Return (mean_BCE, mean_KL) for the whole loader, with no updates."""
    # TODO I.5 (3 points): account for unequal last-batch sizes.
    raise NotImplementedError('TODO I.5')

history=[];start_epoch=0
if CKPT.exists():
    data=torch.load(CKPT,map_location='cpu',weights_only=False)
    if data['latent_dim']!=LATENT_DIM or float(data['beta'])!=BETA:
        raise ValueError('Existing checkpoint has a different latent_dim or beta; change OUT or restore configuration.')
    if int(data.get('train_split_seed',SEED)) != SEED:
        raise ValueError('Checkpoint training split differs from current seed.')
    model.load_state_dict(data['model']); optimizer.load_state_dict(data['optimizer'])
    if USE_AMP and data.get('scaler'):scaler.load_state_dict(data['scaler'])
    history=data['history'];start_epoch=int(data['epoch'])
    previews=data.get('previews',{})
    print('Loaded checkpoint after epoch',start_epoch)
else:
    print('No checkpoint — training from epoch 1')

In [ ]:
# Test the validation loop on eight examples, before training.
_small_loader=DataLoader(TensorDataset(train_tensor[:8]),batch_size=4,shuffle=False)
_chk_bce,_chk_kl=evaluate(_small_loader)
assert np.isfinite(_chk_bce) and np.isfinite(_chk_kl)
print(f'PASS — validation loop: BCE={_chk_bce:.2f}, KL={_chk_kl:.2f}')
del _small_loader

### I.6 — One training step (8 points)

Implement `fashion_train_step(model, optimizer, scaler, x) -> (bce, kl)`, returning Python floats. Reset gradients, run forward and loss, backpropagate, clip gradients to norm 5, and update the optimizer. AMP is supported on CUDA; use the provided scaler to handle it. The test uses a temporary model and permits a few initial skipped AMP updates.


In [ ]:
def fashion_train_step(model, optimizer, scaler, x):
    """One optimizer step on normalized [B,1,28,28] x; return (BCE, KL) as floats."""
    # TODO I.6 (8 points): forward, BCE+KL, backward, grad clipping,
    # optimizer/scaler update. Use autocast only when USE_AMP is True.
    raise NotImplementedError('TODO I.6')


In [ ]:
# Verify an optimizer update on a temporary model (no effect on the trained model).
# With CUDA AMP, GradScaler may skip early steps if it encounters overflow.
probe_model = ConvVAE(LATENT_DIM).to(DEVICE)
probe_optim = torch.optim.Adam(probe_model.parameters(), lr=1e-3)
probe_scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP, init_scale=128.0)
probe_x = torch.rand(8, 1, 28, 28, device=DEVICE)
parameters = [p for p in probe_model.parameters() if p.requires_grad]
initial_parameters = [p.detach().clone() for p in parameters]
updated = False
for attempt in range(8):
    probe_rec, probe_kl = fashion_train_step(
        probe_model, probe_optim, probe_scaler, probe_x
    )
    assert np.isfinite(probe_rec) and np.isfinite(probe_kl), 'Non-finite loss'
    updated = any(not torch.equal(p.detach(), initial)
                  for p, initial in zip(parameters, initial_parameters))
    if updated:
        break
assert updated, ('No weights changed after 8 attempts. Check the train-step TODO, '
                 'learning rate, and whether AMP is repeatedly detecting non-finite gradients.')
print(f'PASS — optimizer updated parameters; BCE={probe_rec:.2f}, KL={probe_kl:.2f}')
del probe_model, probe_optim, probe_scaler, probe_x, parameters, initial_parameters


In [ ]:
from tqdm.auto import tqdm
for epoch in range(start_epoch+1, EPOCHS+1):
    model.train();seen=0; rec_sum=0.;kl_sum=0.
    for (u,) in tqdm(train_loader,desc=f'Epoch {epoch}/{EPOCHS}',leave=False):
        x=u.to(DEVICE,dtype=torch.float32,non_blocking=True)/255.0
        rec_value, kl_value = fashion_train_step(model, optimizer, scaler, x)
        seen+=len(x);rec_sum+=rec_value*len(x);kl_sum+=kl_value*len(x)
    val_bce,val_kl=evaluate(val_loader)
    history.append(dict(epoch=epoch,train_bce=rec_sum/seen,train_kl=kl_sum/seen,
                        val_bce=val_bce,val_kl=val_kl))
    if epoch in PREVIEW_EPOCHS:
        with torch.no_grad(): previews[epoch]=decode_probs(fixed_z[:12]).cpu()
    state={'epoch':epoch,'model':model.state_dict(),'optimizer':optimizer.state_dict(),
           'scaler':scaler.state_dict(),'latent_dim':LATENT_DIM,'beta':BETA,
           'train_split_seed':SEED, 'history':history,'previews':previews}
    temp=CKPT.with_suffix('.tmp');torch.save(state,temp);temp.replace(CKPT)
    if epoch==1 or epoch%10==0 or epoch==EPOCHS:
        print(f'Epoch {epoch:3d}/{EPOCHS} | train BCE={rec_sum/seen:.1f}, KL={kl_sum/seen:.1f}'
              f' | val BCE={val_bce:.1f}, KL={val_kl:.1f}')
if start_epoch>=EPOCHS:print(f'Checkpoint already has {start_epoch} epochs; skipping training.')

if previews:
    epochs = sorted(previews)
    fig, axes=plt.subplots(len(epochs),12,figsize=(14,1.55*len(epochs)),squeeze=False)
    for r, ep in enumerate(epochs):
        for c in range(12):
            ax=axes[r,c];ax.imshow(previews[ep][c,0],cmap='gray',vmin=0,vmax=1);ax.axis('off')
        axes[r,0].set_ylabel(f'Epoch {ep}',rotation=0,labelpad=34,va='center',fontsize=9)
    fig.suptitle('Generation during training · fixed latent vectors',fontsize=11)
    fig.tight_layout(rect=(0,0,1,.94));fig.savefig(OUT/'generation_progress.png',dpi=150,bbox_inches='tight')
    plt.show();plt.close(fig)

### Training progression: one set of fixed latent vectors

A single set of Gaussian latent draws is decoded after selected epochs. The rows correspond to training checkpoints; each **column uses the same latent vector** at every epoch. The saved image and GIF are alternative views of the same experiment.

**Expected output:** clothing silhouettes become increasingly recognizable as the decoder learns. This progression is not a comparison of models with different priors.


In [ ]:
# Assemble the saved decoder previews into a GIF (same fixed z at every epoch).
# Pillow is used only for figure export; the model itself remains PyTorch-only.
from PIL import Image as PILImage, ImageDraw, ImageFont
from IPython.display import display, Image as IPyImage

def export_training_gif(previews, dest, scale=3):
    epochs = sorted(int(e) for e in previews)
    if not epochs:
        print('No preview frames were saved; run training first.')
        return None
    frames=[]
    for ep in epochs:
        examples=previews[ep][:12].cpu().numpy()
        canvas=PILImage.new('RGB',(6*28*scale, 2*28*scale+30),'white')
        draw=ImageDraw.Draw(canvas)
        draw.text((7,7),f'Epoch {ep} · fixed latent vectors',fill='black')
        for j,example in enumerate(examples):
            digit=PILImage.fromarray(np.uint8(np.clip(example[0],0,1)*255),mode='L')
            digit=digit.resize((28*scale,28*scale),resample=PILImage.Resampling.NEAREST).convert('RGB')
            canvas.paste(digit,((j%6)*28*scale,30+(j//6)*28*scale))
        frames.append(canvas)
    # A longer pause on the most recent model helps compare it with the first frame.
    durations=[650]*(len(frames)-1)+[1600]
    frames[0].save(dest,save_all=True,append_images=frames[1:],duration=durations,loop=0,optimize=True)
    print('Saved GIF:',dest,'| frames:',len(frames),'| shared latent vectors:',len(previews[epochs[0]][:12]))
    return dest

gif_path=export_training_gif(previews,OUT/'generation_progress.gif')
if gif_path is not None:
    display(IPyImage(filename=str(gif_path)))

### I.7 — Unconditional sampling (5 points)

Implement `generate_from_prior(model, n, latent_dim, device, seed) -> images` with shape `[n,1,28,28]` on CPU and values in `[0,1]`. Draw **independent** latent vectors from \(\mathcal N(0,I)\), decode, and apply `sigmoid`. No input image or class label is provided.

**Expected figure:** 64 independent prior samples; the images need not represent any requested class. Unlike reconstructions, these are entirely new latent draws.


In [ ]:
@torch.no_grad()
def generate_from_prior(model, n, latent_dim, device, seed=2718):
    """Return CPU probabilities [n,1,28,28] decoded from fresh N(0,I) draws."""
    # TODO I.7 (5 points): use the supplied seed and a CPU Generator.
    raise NotImplementedError('TODO I.7')


In [ ]:
# These images are decoded independent prior draws. They are not reconstructions.
new_images=generate_from_prior(model,64,LATENT_DIM,DEVICE,seed=2718)
assert new_images.shape == (64,1,28,28)
assert 0 <= float(new_images.min()) and float(new_images.max()) <= 1
image_grid(new_images, ncol=8,
           title='Unconditional samples · 64 draws from N(0, I)',
           save_as='prior_samples.png')

### Reconstruction is a different task

For each **held-out test** image, display the original image above its reconstruction from the posterior **mean**. This measures encoder–decoder reconstruction. It is **not** unconditional generation.

In [ ]:
with torch.no_grad():
    model.eval()
    x_test = test_tensor[:12].to(DEVICE,dtype=torch.float32)/255
    mu,_ = model.encode(x_test)
    recon = torch.sigmoid(model.decode(mu)).cpu()
    originals=x_test.cpu()

fig,axes=plt.subplots(2,12,figsize=(14,2.9),squeeze=False)
for j in range(12):
    axes[0,j].imshow(originals[j,0],cmap='gray',vmin=0,vmax=1)
    axes[1,j].imshow(recon[j,0],cmap='gray',vmin=0,vmax=1)
    for r in range(2):axes[r,j].axis('off')
for r,label in enumerate(('Test image','Reconstruction')):
    axes[r,0].set_ylabel(label,rotation=0,labelpad=52,fontsize=10,va='center')
fig.suptitle('Test images and posterior-mean reconstructions',fontsize=11)
fig.tight_layout(rect=(0,0,1,.92));fig.savefig(OUT/'test_reconstructions.png',dpi=160,bbox_inches='tight')
plt.show();plt.close(fig)


### I.8 — Interpolation (5 points)

Implement `interpolation_paths(model, image_a, image_b, steps) -> (pixel_path, latent_path)`, returning two CPU tensors of shape `[steps, 1, 28, 28]`.

- Encode the two held-out images using their posterior **means**.
- **Pixel path:** Linearly blend the decoded reconstructions of the two images.
- **Latent path:** Linearly interpolate between the two latent means and decode each point.
- Include both endpoints. The two paths must start and end with the same reconstructed images.

The resulting figure compares pixel blending and latent interpolation between the same endpoints.


In [ ]:
@torch.no_grad()
def interpolation_paths(model, image_a, image_b, steps=9):
    """Return (pixel_path, latent_path), CPU tensors [steps,1,28,28]."""
    # TODO I.8 (5 points): both paths must share the decoded endpoints.
    raise NotImplementedError('TODO I.8')


In [ ]:
# Labels are used only to choose and name held-out visual examples.
CLASS_NAMES = ['T-shirt/top','Trouser','Pullover','Dress','Coat','Sandal',
               'Shirt','Sneaker','Bag','Ankle boot']
PAIR_CLASSES = [(9,1),(8,4),(7,3)]  # Ankle boot → Trouser; Bag → Coat; Sneaker → Dress

for class_a,class_b in PAIR_CLASSES:
    ia=int(np.flatnonzero(test_labels==class_a)[0]); ib=int(np.flatnonzero(test_labels==class_b)[0])
    xa=test_tensor[ia].float().div(255); xb=test_tensor[ib].float().div(255)
    pixel_path,latent_path=interpolation_paths(model,xa,xb,steps=9)
    assert pixel_path.shape == (9,1,28,28) and latent_path.shape == (9,1,28,28)
    assert torch.allclose(pixel_path[[0,-1]],latent_path[[0,-1]],atol=1e-5), 'Paths must share endpoints.'
    fig, axes=plt.subplots(2,9,figsize=(13,3.6),squeeze=False)
    for j in range(9):
        axes[0,j].imshow(pixel_path[j,0],cmap='gray',vmin=0,vmax=1)
        axes[1,j].imshow(latent_path[j,0],cmap='gray',vmin=0,vmax=1)
        for r in range(2):axes[r,j].axis('off')
        axes[1,j].set_xlabel(f'{j/8:.2f}',fontsize=9)
    axes[0,0].set_ylabel('Pixel blend',rotation=0,labelpad=46,va='center',fontsize=10)
    axes[1,0].set_ylabel('Latent path',rotation=0,labelpad=46,va='center',fontsize=10)
    fig.suptitle(f'{CLASS_NAMES[class_a]} → {CLASS_NAMES[class_b]} · shared endpoints · α: 0 to 1',fontsize=12)
    fig.tight_layout(rect=(0,0,1,.93))
    file_name=f'interpolation_{class_a}_to_{class_b}.png'
    fig.savefig(OUT/file_name,dpi=150,bbox_inches='tight')
    plt.show();plt.close(fig)
    print('A:',CLASS_NAMES[class_a],'test index',ia,'| B:',CLASS_NAMES[class_b],'test index',ib)


### I.9 — Training results

Plot the training and validation BCE/KL across epochs. Report the held-out test BCE and KL after training.

BCE and KL are summed over pixels and latent dimensions, respectively, and then averaged **per image**. Report both values in nats/image.


In [ ]:
import pandas as pd
h=pd.DataFrame(history)
fig,axs=plt.subplots(1,2,figsize=(10,3.3),constrained_layout=True)
axs[0].plot(h.epoch,h.train_bce,label='Training')
axs[0].plot(h.epoch,h.val_bce,label='Validation')
axs[0].set(title='Reconstruction BCE',xlabel='Epoch',ylabel='BCE ↓')
axs[0].legend()
axs[1].plot(h.epoch,h.train_kl,label='Training')
axs[1].plot(h.epoch,h.val_kl,label='Validation')
axs[1].set(title='Gaussian KL',xlabel='Epoch',ylabel='KL')
axs[1].legend()
fig.savefig(OUT/'bce_kl.png',dpi=150)
plt.show();plt.close(fig)

test_bce,test_kl=evaluate(test_loader)
print(f'Held-out test: BCE={test_bce:.2f} nats/image | KL={test_kl:.2f} nats/image')
print('Figures and checkpoint:',OUT)

---

# **Part II — Gaussian and mixture priors (15 points)**

**Goal.** Compare where two VAEs generate observations when the data consist of eight separated modes.

A standard Gaussian prior is simple but may not place enough generated mass near every mode. The mixture-prior model starts with **eight learnable centers arranged on a ring**: that initialization supplies useful structure, which must be acknowledged in the interpretation. Both models have the same encoder, decoder, and training budget.

The synthetic two-dimensional data are provided. You will implement the mixture log-density, the Monte Carlo KL term, and a measure of how many samples fall outside the eight modes.



In [ ]:
import math, os, random, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED=19
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type=='cpu': torch.set_num_threads(min(4, os.cpu_count() or 1))
QUICK_TEST=False  # For debugging only; do not submit QUICK_TEST results
EPOCHS=3 if QUICK_TEST else 90
N_TRAIN=1024 if QUICK_TEST else 10240
N_GENERATED=1500 if QUICK_TEST else 5000
OUT_DIR=Path('/kaggle/working/vae_homework_2026_prior' if Path('/kaggle/working').exists() else './vae_homework_2026_prior')
OUT_DIR.mkdir(parents=True,exist_ok=True)
print('device:',DEVICE,'| epochs:',EPOCHS,'| simulated observations:',N_TRAIN)

### II.1 — Visualize the target distribution

Each point in the plot below is a **2D observation**, not a latent code. The crosses mark the true centers. There are eight separated modes. The observation labels are used only to generate reference samples; the models never receive component labels.

In [ ]:
N_MODES = 8
MODE_RADIUS = 3.0
DATA_STD = 0.14
angles = np.linspace(0, 2*np.pi, N_MODES, endpoint=False)
TRUE_CENTERS = np.stack([MODE_RADIUS*np.cos(angles), MODE_RADIUS*np.sin(angles)], 1).astype(np.float32)

def eight_modes(n, seed):
    local_rng = np.random.default_rng(seed)
    ids = local_rng.integers(0, N_MODES, n)
    return (TRUE_CENTERS[ids] + local_rng.normal(scale=DATA_STD, size=(n,2))).astype(np.float32)

toy_train = torch.from_numpy(eight_modes(N_TRAIN, SEED+110))
toy_test = torch.from_numpy(eight_modes(2048, SEED+111))
toy_loader = DataLoader(TensorDataset(toy_train), batch_size=256, shuffle=True)
fig, ax = plt.subplots(figsize=(4.7,4.7))
ax.scatter(toy_train[:3500,0], toy_train[:3500,1], s=3, alpha=.35)
ax.scatter(TRUE_CENTERS[:,0], TRUE_CENTERS[:,1], c='black', marker='x', s=55)
ax.set(aspect='equal', xlim=(-4.3,4.3), ylim=(-4.3,4.3), title='Observed 2D data | eight Gaussian modes')
plt.tight_layout(); plt.show()
print('Expected: eight distinct clusters, with empty gaps between them.')

### II.2 — Mixture log-density (5 points)

Implement `mixture_log_prob(z, centers, component_std) -> [B]` for \(K\) equally weighted, isotropic Gaussian components of dimension \(D\):

$$\log p(z)=\log\!\left[\frac1K\sum_{k=1}^{K}\mathcal N(z;\mu_k,s^2I)\right].$$

`z` has shape `[B,D]`, `centers` has shape `[K,D]`, and `component_std` is a positive scalar. Use `torch.logsumexp` for numerical stability. Each output is one latent log-density. The centers remain trainable.


In [ ]:
def mixture_log_prob(z, centers, component_std):
    """Return per-example log-density [B] for equally weighted Gaussian mixture."""
    # TODO II.2 (5 points): use torch.logsumexp.
    raise NotImplementedError('TODO II.2')


### II.3 — Monte Carlo KL objective (4 points)

Implement `toy_vae_loss(x, pred, mu, logvar, z, model, obs_std=0.14) -> (loss, rec, kl)`:

$$\widehat{D}_{\mathrm{KL}} = \frac1B\sum_{b=1}^B\Big[\log q_\phi(z_b\mid x_b)-\log p(z_b)\Big].$$

Use the log-density of the **diagonal Gaussian** \(q_\phi(z\mid x)\), including its normalizing constant. `model.log_prior(z)` handles either prior. The reconstruction part is \(\|x-\hat x\|_2^2/(2s_x^2)\), summed across observation dimensions and averaged over the batch. All three outputs are scalar tensors and must remain differentiable.


In [ ]:
class ToyVAE(nn.Module):
    def __init__(self, prior='normal', latent_dim=2, component_std=0.65):
        super().__init__()
        assert prior in ('normal','mixture')
        self.prior = prior
        self.latent_dim = latent_dim
        self.component_std = component_std
        self.encoder = nn.Sequential(nn.Linear(2,96), nn.Tanh(), nn.Linear(96,96), nn.Tanh())
        self.mu = nn.Linear(96, latent_dim)
        self.logvar = nn.Linear(96, latent_dim)
        self.decoder = nn.Sequential(nn.Linear(latent_dim,96), nn.Tanh(),
                                     nn.Linear(96,96), nn.Tanh(), nn.Linear(96,2))
        if prior == 'mixture':
            phase = torch.linspace(0, 2*math.pi, N_MODES+1)[:-1]
            initial = torch.stack([2.0*torch.cos(phase), 2.0*torch.sin(phase)], dim=-1)
            self.prior_centers = nn.Parameter(initial)

    def encode(self,x):
        h = self.encoder(x)
        return self.mu(h), self.logvar(h).clamp(-12, 8)

    def forward(self,x):
        mu, lv = self.encode(x)
        z = mu+torch.exp(0.5*lv)*torch.randn_like(mu)
        return self.decoder(z), mu, lv, z

    def log_prior(self,z):
        if self.prior == 'mixture':
            return mixture_log_prob(z, self.prior_centers, self.component_std)
        return -0.5*(z.square()+math.log(2*math.pi)).sum(1)

    @torch.no_grad()
    def sample(self,n, obs_std=0.14):
        if self.prior == 'mixture':
            idx = torch.randint(N_MODES, (n,), device=self.prior_centers.device)
            z = self.prior_centers[idx] + self.component_std*torch.randn(n,self.latent_dim, device=self.prior_centers.device)
        else:
            z = torch.randn(n,self.latent_dim,device=next(self.parameters()).device)
        return self.decoder(z) + obs_std*torch.randn(n,2,device=z.device)

def toy_vae_loss(x, pred, mu, logvar, z, model, obs_std=0.14):
    """Return (total, reconstruction, MC KL); scalar differentiable tensors."""
    # TODO II.3 (4 points): compute log_q(z|x) and log_p(z) as densities.
    raise NotImplementedError('TODO II.3')

In [ ]:
# Mathematical sanity checks on density and loss, before fitting.
toy_z = torch.tensor([[0.,0.],[1.,-1.]])
centers_check = torch.tensor([[0.,0.]])
standard_check = -0.5*(toy_z.square()+math.log(2*math.pi)).sum(1)
assert torch.allclose(mixture_log_prob(toy_z, centers_check, 1.), standard_check, atol=1e-5)
toy_check = ToyVAE(prior='mixture').to(DEVICE)
xx = toy_train[:6].to(DEVICE)
pp,mm,ll,zz = toy_check(xx)
trial_loss, _, _ = toy_vae_loss(xx,pp,mm,ll,zz,toy_check)
assert torch.isfinite(trial_loss)
trial_loss.backward()
assert toy_check.prior_centers.grad is not None
print('PASS — II.2: mixture density and differentiable VAE loss')

### II.4 — Fit the two priors

The training loop is provided. It optimizes a Gaussian observation likelihood and the Monte Carlo estimate of the KL. Both models receive **only 2D observations**, never the generating mode index.

**Output:** a log of the reconstruction and KL terms, plus two trained models. The objective values of different priors should not be treated as a direct measure of sample quality.


In [ ]:
def fit_toy(prior, epochs=EPOCHS):
    torch.manual_seed(SEED+210)  # controlled initialization for shared network weights
    model = ToyVAE(prior=prior).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=1.5e-3)
    logs = {k: [] for k in ('total', 'reconstruction', 'kl', 'validation')}
    for epoch in range(epochs):
        model.train()
        running = np.zeros(3); count = 0
        for (xb,) in toy_loader:
            xb = xb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            pred, mu, lv, z = model(xb)
            loss, rec, kl = toy_vae_loss(xb, pred, mu, lv, z, model)
            loss.backward(); opt.step()
            running += len(xb)*np.array([loss.item(), rec.item(), kl.item()])
            count += len(xb)
        for key, val in zip(('total', 'reconstruction', 'kl'), running/count):
            logs[key].append(float(val))
        model.eval()
        with torch.no_grad():
            vals = []
            for chunk in toy_test.split(512):
                xb = chunk.to(DEVICE)
                p,m,lv,z = model(xb)
                vl,_,_ = toy_vae_loss(xb,p,m,lv,z,model)
                vals.append(vl.item()*len(xb))
            logs['validation'].append(sum(vals)/len(toy_test))
        if epoch == 0 or (epoch+1) % max(1, epochs//4) == 0:
            print(f'{prior:7s} epoch {epoch+1:3d}/{epochs} '
                  f'total={logs["total"][-1]:.2f} rec={logs["reconstruction"][-1]:.2f} '
                  f'KL≈{logs["kl"][-1]:.2f} val≈{logs["validation"][-1]:.2f}')
    return model.eval(), logs

start = time.perf_counter()
toy_models, toy_logs = {}, {}
for prior in ['normal','mixture']:
    toy_models[prior],toy_logs[prior] = fit_toy(prior)
    torch.save(toy_models[prior].state_dict(), OUT_DIR/f'toy_vae_{prior}.pt')
print(f'Toy training took {(time.perf_counter()-start)/60:.1f} min')

### II.5 — Inspect distributions sampled from each model

The columns show **observations from the reference distribution**, **new observations generated from the Normal-prior VAE**, and **new observations generated from the mixture-prior VAE**, respectively. A dot is one observation. All axes and sample counts are matched. Look for generated points in regions where true observations are rare.

In [ ]:
@torch.no_grad()
def draw_samples(model,n):
    model.eval()
    return model.sample(n,obs_std=DATA_STD).cpu().numpy()

reference=eight_modes(N_GENERATED,SEED+1111)
pred_normal=draw_samples(toy_models['normal'],N_GENERATED)
pred_mixture=draw_samples(toy_models['mixture'],N_GENERATED)
clouds=[('Reference: real observations',reference),('Generated: Normal prior',pred_normal),('Generated: mixture prior',pred_mixture)]
fig,axs=plt.subplots(1,3,figsize=(13.3,4.6),sharex=True,sharey=True)
for ax,(title,pts) in zip(axs,clouds):
    ax.scatter(pts[:,0],pts[:,1],s=3,alpha=.28)
    ax.scatter(TRUE_CENTERS[:,0],TRUE_CENTERS[:,1],s=50,marker='x',c='black')
    ax.set(aspect='equal',xlim=(-4.5,4.5),ylim=(-4.5,4.5),title=title,xlabel='Observed x')
axs[0].set_ylabel('Observed y')
fig.suptitle('Observed distribution versus independently generated observations',fontsize=13)
fig.tight_layout()
fig.savefig(OUT_DIR/'prior_distribution_comparison.png',dpi=160)
plt.show()

### II.6 — Off-mode diagnostic (3 points)

Implement `mode_diagnostics(samples, centers, radius=0.45) -> dict` with keys:

- `off_mode_fraction`: fraction of generated observations at distance **at least** `radius` from every true center.
- `covered_modes`: number of centers assigned at least 1% of all generated observations *within* the radius.
- `mean_nearest_distance`: mean Euclidean distance to the closest true center.

Use `[N,2]` arrays of observations and `[8,2]` arrays of centers. The radius is specific to this synthetic task, not a general generative-model metric.


In [ ]:
def mode_diagnostics(samples, centers=TRUE_CENTERS, radius=0.45):
    """Return dict: off_mode_fraction, covered_modes, mean_nearest_distance."""
    # TODO II.6 (3 points): nearest-center Euclidean distances and mode counts.
    raise NotImplementedError('TODO II.6')


In [ ]:
# Unit test: one point at a true center, one remote point.
check_points = np.array([TRUE_CENTERS[0], [12., 12.]], dtype=np.float32)
check = mode_diagnostics(check_points, radius=0.45)
assert abs(check['off_mode_fraction'] - .5) < 1e-9
assert check['covered_modes'] == 1
assert check['mean_nearest_distance'] > 1
print('PASS — II.5: distances, off-mode count and coverage')

In [ ]:
scores={name:mode_diagnostics(points) for name,points in clouds}
for name,m in scores.items():
    print(f'{name:22s} | off-mode {m["off_mode_fraction"]*100:5.1f}% | '
          f'covered {m["covered_modes"]}/8 | nearest distance {m["mean_nearest_distance"]:.3f}')



### II.7 — Discussion (3 points)

Use your generated samples and measured results to answer the following questions.

1. Which model generates more samples outside the eight modes? Report the off-mode fraction for both models. (2 points)

2. The mixture prior starts with eight centers arranged in a circle. How might this starting point affect the comparison between the two models? (1 point)

Write your answers below.


---

# **Part III — Denoising VAE on MNIST (45 points)**

**Question.** Does training an encoder on incomplete images improve the restoration of missing pixels, and what happens to its reconstruction accuracy on complete images?

Two convolutional VAEs use the same architecture, clean reconstruction targets, and β-weighted training schedule. Only the **encoder inputs and masks** differ. Report results on held-out MNIST using paired masks. For reproducibility, this experiment retains **β=0.25**, with a four-epoch warm-up. It is not the standard β=1 ELBO.

**Deliverables:** observation mask checks, clean reconstructions, restorations under scattered and square missingness, and masked-pixel MSE curves.

### III.1 — MNIST data

The loader downloads and caches MNIST. It creates **30,000 training, 2,000 validation, and 2,000 test images**, with disjoint training/validation indices. The test split is never used to select a model. Labels are used only for choosing examples for the figures, not as inputs to either VAE.

**Output:** dataset source and DataLoader sizes.


In [ ]:
# Configuration for Part III only; no effect on Parts A or B.
import urllib.request
C_SEED = 27
C_DEVICE = DEVICE
C_EPOCHS = 18
C_BATCH = 128
C_LATENT_DIM = 20
C_BETA = 0.25
C_WARMUP_EPOCHS = 4
C_N_TRAIN, C_N_VAL, C_N_TEST = 30_000, 2_000, 2_000
C_DIR = Path('./vae_homework_2026_restoration')
C_DIR.mkdir(parents=True, exist_ok=True)
C_CHECKPOINT_DIR = C_DIR / 'checkpoints'
C_CHECKPOINT_DIR.mkdir(exist_ok=True)
print(f'Part III | device: {C_DEVICE} | beta: {C_BETA} | epochs/model: {C_EPOCHS}')

In [ ]:
C_MNIST_URL = 'https://storage.googleapis.com/tensorflow/tf-keras-datasets/mnist.npz'

def c_validate_mnist(path):
    with np.load(path, allow_pickle=False) as data:
        required=('x_train','y_train','x_test','y_test')
        if not all(k in data.files for k in required):
            raise ValueError('Not a valid MNIST archive: keys missing')
        arrays=tuple(data[k].copy() for k in required)
    tr,ytr,te,yte=arrays
    assert tr.shape==(60000,28,28) and te.shape==(10000,28,28)
    assert tr.dtype==np.uint8 and te.dtype==np.uint8
    assert len(ytr)==60000 and len(yte)==10000
    return arrays

def c_load_mnist():
    choices=[]
    override=os.environ.get('MNIST_NPZ_OVERRIDE','')
    if override:choices.append(Path(override))
    choices.extend([Path('data/mnist.npz'), Path.home()/'.keras/datasets/mnist.npz'])
    for path in choices:
        if path.is_file():
            try:return (*c_validate_mnist(path),str(path))
            except (ValueError,AssertionError,OSError):pass
    path=Path('data/mnist.npz');path.parent.mkdir(exist_ok=True,parents=True)
    temporary=path.with_suffix('.partial')
    print('Downloading MNIST once (about 11 MB)...')
    try:
        with urllib.request.urlopen(C_MNIST_URL,timeout=30) as response, open(temporary,'wb') as writer:
            import shutil
            shutil.copyfileobj(response,writer)
        arrays=c_validate_mnist(temporary)
        temporary.replace(path)
        return (*arrays,str(path))
    except Exception as exc:
        temporary.unlink(missing_ok=True)
        raise RuntimeError('MNIST download failed. Connect to the Internet once or set MNIST_NPZ_OVERRIDE to a local mnist.npz.') from exc

c_xtr,c_ytr,c_xte,c_yte,c_data_source=c_load_mnist()
c_rng=np.random.default_rng(C_SEED)
c_perm=c_rng.permutation(len(c_xtr))
c_train_ids,c_val_ids=c_perm[:C_N_TRAIN],c_perm[C_N_TRAIN:C_N_TRAIN+C_N_VAL]
c_test_ids=c_rng.permutation(len(c_xte))[:C_N_TEST]
assert np.intersect1d(c_train_ids,c_val_ids).size == 0

def c_as_tensors(x,y,indices):
    return (torch.from_numpy(np.ascontiguousarray(x[indices])).unsqueeze(1).float()/255.,
            torch.from_numpy(np.ascontiguousarray(y[indices])).long())

c_train_x,c_train_y=c_as_tensors(c_xtr,c_ytr,c_train_ids)
c_val_x,c_val_y=c_as_tensors(c_xtr,c_ytr,c_val_ids)
c_test_x,c_test_y=c_as_tensors(c_xte,c_yte,c_test_ids)
c_train_loader=DataLoader(TensorDataset(c_train_x,c_train_y),batch_size=C_BATCH,shuffle=True,num_workers=0)
c_val_loader=DataLoader(TensorDataset(c_val_x,c_val_y),batch_size=C_BATCH,shuffle=False,num_workers=0)
c_test_loader=DataLoader(TensorDataset(c_test_x,c_test_y),batch_size=C_BATCH,shuffle=False,num_workers=0)
print('MNIST source:',c_data_source)
print('Train / validation / test:',len(c_train_x),len(c_val_x),len(c_test_x))
print('Label conditioning: none')

### III.2 — Observation masks (9 points)

A mask \(m\) is 1 at an observed pixel and 0 at a missing pixel. The encoder receives both the corrupted image and its mask, because a black pixel and an unobserved pixel are not interchangeable.

Implement `c_corrupt(clean, kind, severity) -> (observed, mask)`:

- `clean`: `[B,1,28,28]`, floats in `[0,1]`.
- `dropout`: independently hide each pixel with probability `severity` (*input corruption*, not network dropout).
- `square`: hide one randomly positioned square **per image**, of side `round(28 * severity)`.
- `clean`: return the input unchanged with an all-ones mask.

Both return tensors must match the input shape; `observed == clean * mask`. The unit test checks mask values and dimensions before training.


In [ ]:
def c_corrupt(clean, kind='dropout', severity=0.5):
    """Return (corrupted, mask), both same shape as clean; mask=1 means observed."""
    # TODO III.2 (9 points): independent dropout, a random square per image,
    # and an identity ('clean') condition. Preserve dtype and device.
    raise NotImplementedError('TODO III.2')


In [ ]:
# Shape and integrity checks. No model training is required.
c_ones=torch.ones(64,1,28,28)
for c_kind,c_s in [('clean',0.),('dropout',0.5),('square',0.4)]:
    c_observed,c_mask=c_corrupt(c_ones,c_kind,c_s)
    assert c_observed.shape == c_mask.shape == c_ones.shape
    assert torch.all((c_mask==0)|(c_mask==1))
    assert torch.allclose(c_observed,c_ones*c_mask)
    if c_kind=='clean':assert torch.all(c_mask==1)
    if c_kind=='dropout':assert 0.35 < 1-c_mask.mean().item() < 0.65
    print(f'{c_kind:7s} | fraction missing: {1-c_mask.mean().item():.3f}')
print('PASS — observation masks')

### III.3 — Corruption schedule (5 points)

Implement `c_training_view(clean) -> (observed, mask)` using these probabilities for each training **batch**:

| Input | Probability | Severity |
|---|---:|---|
| Unchanged | 0.15 | — |
| Random pixel masking | 0.50 | 20–65% hidden |
| Square occlusion | 0.35 | side 25–50% of image width |

Sample the severity uniformly from the stated interval, then call `c_corrupt`. The clean VAE instead always receives its complete input and an all-ones mask. Both models target the same clean image.


In [ ]:
def c_training_view(clean):
    """Draw one training corruption regime and return (observed, mask)."""
    # TODO III.3 (5 points): use stated probabilities and severity intervals.
    raise NotImplementedError('TODO III.3')


In [ ]:
c_demo=torch.ones(16,1,28,28)
for _ in range(3):
    obs,msk=c_training_view(c_demo)
    assert obs.shape==msk.shape==c_demo.shape
    assert torch.allclose(obs,c_demo*msk)
print('PASS — corruption schedule returns valid observations')


### III.4 — Train matched models (7 points)

The two VAEs have identical encoder/decoder architectures, a 20-dimensional Gaussian latent space, and an identical optimizer schedule. The encoder receives a two-channel tensor `(observed image, mask)`.

Implement `c_run_epoch(net, loader, optimizer, denoising, beta) -> (bce, kl)`:

- With an optimizer, train the model. For the denoising model, sample a new corrupted observation with `c_training_view` each batch; the clean model receives a complete image.
- Evaluate with `optimizer=None` on clean inputs; do not update weights.
- The likelihood target is **always the clean image**. Return mean per-image reconstruction BCE and KL.

The objective is $\mathrm{BCE} + \beta\,\mathrm{KL}$. Preserve the original experiment's four-epoch warm-up to **$\beta=0.25$**. This is *not* the standard $\beta=1$ ELBO, and the two sections have different objectives by design.


In [ ]:
class CEncoder(nn.Module):
    def __init__(self,dim=C_LATENT_DIM):
        super().__init__()
        self.conv=nn.Sequential(nn.Conv2d(2,32,4,2,1),nn.SiLU(),
                                nn.Conv2d(32,64,4,2,1),nn.SiLU(),nn.Flatten())
        self.mean=nn.Linear(64*7*7,dim)
        self.logvar=nn.Linear(64*7*7,dim)
    def forward(self,observed,mask):
        h=self.conv(torch.cat((observed,mask),dim=1))
        return self.mean(h),self.logvar(h).clamp(-12,8)

class CDecoder(nn.Module):
    def __init__(self,dim=C_LATENT_DIM):
        super().__init__()
        self.fc=nn.Sequential(nn.Linear(dim,64*7*7),nn.SiLU())
        self.deconv=nn.Sequential(nn.ConvTranspose2d(64,64,4,2,1),nn.SiLU(),
                                  nn.ConvTranspose2d(64,32,4,2,1),nn.SiLU(),
                                  nn.Conv2d(32,1,3,padding=1))
    def forward(self,z):
        h=self.fc(z).reshape(-1,64,7,7)
        return self.deconv(h)

class CConvVAE(nn.Module):
    def __init__(self,dim=C_LATENT_DIM):
        super().__init__()
        self.encoder=CEncoder(dim)
        self.decoder=CDecoder(dim)
    def forward(self,observed,mask):
        mu,logvar=self.encoder(observed,mask)
        return self.decoder(reparameterize(mu,logvar)),mu,logvar
    @torch.no_grad()
    def reconstruct(self,observed,mask):
        mu,_=self.encoder(observed,mask)
        return self.decoder(mu).sigmoid()

c_net=CConvVAE().to(C_DEVICE)
c_probe=torch.ones(4,1,28,28,device=C_DEVICE)
c_logits,c_mu,c_lv=c_net(c_probe,torch.ones_like(c_probe))
assert c_logits.shape==c_probe.shape and c_mu.shape==(4,C_LATENT_DIM)
assert torch.isfinite(c_logits).all() and torch.isfinite(c_lv).all()
print('PASS — restoration VAE dimensions and finite outputs')

In [ ]:
def c_reconstruction_nll(logits,clean):
    return F.binary_cross_entropy_with_logits(logits,clean,reduction='none').flatten(1).sum(1).mean()

def c_run_epoch(net, loader, optimizer, denoising, beta):
    """Return mean (BCE, KL) over an epoch; optimizer=None means evaluation."""
    # TODO III.4 (7 points): clean-target likelihood, corruption on training
    # batches only for denoising model, beta-weighted KL, update if training.
    raise NotImplementedError('TODO III.4')

# Check the one-epoch interface before the full run.
_c_toy_net=CConvVAE().to(C_DEVICE)
_c_toy_opt=torch.optim.Adam(_c_toy_net.parameters(),lr=1e-3)
_c_toy_data=DataLoader(TensorDataset(c_train_x[:8],c_train_y[:8]),batch_size=4,shuffle=False)
_c_bce,_c_kl=c_run_epoch(_c_toy_net,_c_toy_data,_c_toy_opt,True,C_BETA/4)
assert np.isfinite(_c_bce) and np.isfinite(_c_kl)
print(f'PASS — restoration training loop: BCE={_c_bce:.1f}, KL={_c_kl:.1f}')
del _c_toy_net,_c_toy_opt,_c_toy_data

# Both models have the same architecture, LR, split, and warm-up schedule.
c_models,c_histories={},{}
for c_name,c_denoising in [('Clean VAE',False),('Denoising VAE',True)]:
    torch.manual_seed(C_SEED+int(c_denoising))
    c_net=CConvVAE().to(C_DEVICE)
    c_optim=torch.optim.Adam(c_net.parameters(),lr=1.5e-3)
    c_file=C_CHECKPOINT_DIR/(('denoising' if c_denoising else 'clean')+'_beta025.pt')
    c_start=0;c_history=[]
    if c_file.is_file():
        c_state=torch.load(c_file,map_location='cpu',weights_only=False)
        if (c_state['beta'],c_state['epochs_config'],c_state['latent_dim']) != (C_BETA,C_EPOCHS,C_LATENT_DIM):
            raise RuntimeError('Part III checkpoint configuration mismatch. Use a new output directory.')
        c_net.load_state_dict(c_state['model'])
        c_optim.load_state_dict(c_state['optimizer'])
        c_start=c_state['epoch'];c_history=c_state['history']
        print(f'{c_name}: resume after epoch {c_start}')
    for c_epoch in range(c_start+1,C_EPOCHS+1):
        c_beta=C_BETA*min(1.,c_epoch/C_WARMUP_EPOCHS)
        c_train_rec,c_train_kl=c_run_epoch(c_net,c_train_loader,c_optim,c_denoising,c_beta)
        c_val_rec,c_val_kl=c_run_epoch(c_net,c_val_loader,None,False,c_beta)
        c_history.append(dict(epoch=c_epoch,beta=c_beta,
                              train_bce=c_train_rec,train_kl=c_train_kl,
                              val_bce=c_val_rec,val_kl=c_val_kl))
        c_state={'model':c_net.state_dict(),'optimizer':c_optim.state_dict(),
                 'epoch':c_epoch,'history':c_history,'beta':C_BETA,
                 'epochs_config':C_EPOCHS,'latent_dim':C_LATENT_DIM}
        c_tmp=c_file.with_suffix('.tmp');torch.save(c_state,c_tmp);c_tmp.replace(c_file)
        if c_epoch==1 or c_epoch%3==0 or c_epoch==C_EPOCHS:
            print(f'{c_name:13s} | epoch {c_epoch:02d}/{C_EPOCHS} | '
                  f'train BCE {c_train_rec:.1f}, KL {c_train_kl:.1f} | '
                  f'clean val BCE {c_val_rec:.1f}')
    c_models[c_name]=c_net.eval()
    c_histories[c_name]=c_history
print('PASS — Part III model training and checkpoint saving')

### III.5 — Clean reconstruction

Both models reconstruct **the same held-out clean digits** through their posterior means. Report test BCE and KL for each model. This test evaluates reconstruction, **not** generation from the prior.

**Figure:** rows are the originals, the clean-trained VAE, and the denoising-trained VAE; columns correspond to digits 0–9.


In [ ]:
for c_name,c_net in c_models.items():
    c_bce,c_kl=c_run_epoch(c_net,c_test_loader,None,False,C_BETA)
    print(f'{c_name:13s} | held-out CLEAN BCE {c_bce:.2f} nats/image | KL {c_kl:.2f}')
c_display_ids=[]
for c_digit in range(10):
    c_matches=(c_test_y==c_digit).nonzero().flatten()
    assert len(c_matches), 'Increase C_N_TEST so every digit appears.'
    c_display_ids.append(int(c_matches[0]))
c_display_x=c_test_x[c_display_ids].to(C_DEVICE)
@torch.no_grad()
def c_show_clean():
    images=[c_display_x.cpu()]
    for net in c_models.values():
        images.append(net.reconstruct(c_display_x,torch.ones_like(c_display_x)).cpu())
    fig,axs=plt.subplots(3,10,figsize=(12.5,3.9),squeeze=False)
    names=['Original','Clean VAE','Denoising VAE']
    for r,row in enumerate(images):
        for j in range(10):
            axs[r,j].imshow(row[j,0],cmap='gray',vmin=0,vmax=1);axs[r,j].axis('off')
            if r==0:axs[r,j].set_title(str(j),fontsize=9)
        axs[r,0].text(-.12,.5,names[r],transform=axs[r,0].transAxes,ha='right',va='center',fontsize=8)
    fig.suptitle('Clean-input reconstruction: standard and denoising VAEs',fontsize=11)
    fig.tight_layout(rect=(.12,0,1,.95));fig.savefig(C_DIR/'clean_test_reconstructions.png',dpi=150,bbox_inches='tight')
    plt.show();plt.close(fig)
c_show_clean()

### III.6 — Fill missing pixels (6 points)

Implement `c_restore(net, observed, mask) -> completed`, with all tensors `[B,1,28,28]`. Use the network's posterior-mean reconstruction to fill **only** the missing pixels. Preserve known pixels exactly.

The figures compare both models on identical corruptions: 50% random pixel masking and a square whose side is 40% of the image width. Each column is a held-out digit; each row is labeled.


In [ ]:
@torch.no_grad()
def c_restore(net, observed, mask):
    """Return completed images with observed pixels exactly unchanged."""
    # TODO III.6 (6 points): use net.reconstruct() only for missing values.
    raise NotImplementedError('TODO III.6')


In [ ]:
c_example,c_exmask=c_corrupt(c_display_x,'dropout',0.50)
c_result=c_restore(c_models['Denoising VAE'],c_example,c_exmask)
assert c_result.shape==c_example.shape
assert torch.allclose(c_result*c_exmask,c_example*c_exmask,atol=1e-7)
print('PASS — restoration preserves every observed pixel')

@torch.no_grad()
def c_restoration_grid(kind,severity):
    torch.manual_seed(900 if kind=='dropout' else 901)
    observed,mask=c_corrupt(c_display_x,kind,severity)
    rows=[c_display_x,observed,
          c_restore(c_models['Clean VAE'],observed,mask),
          c_restore(c_models['Denoising VAE'],observed,mask)]
    labels=['Original','Damaged','Clean VAE fill','Denoising VAE fill']
    fig,axs=plt.subplots(4,10,figsize=(12.5,5.1),squeeze=False)
    for r,images in enumerate(rows):
        for j in range(10):
            axs[r,j].imshow(images[j,0].cpu(),cmap='gray',vmin=0,vmax=1);axs[r,j].axis('off')
            if r==0:axs[r,j].set_title(str(j),fontsize=9)
        axs[r,0].text(-.12,.5,labels[r],transform=axs[r,0].transAxes,ha='right',va='center',fontsize=8)
    severity_text=(f'{severity:.0%} pixels independently removed' if kind=='dropout'
                   else f'square side = {severity:.0%} of image width')
    fig.suptitle(f'MNIST restoration: {severity_text} (shared masks)',fontsize=11)
    fig.tight_layout(rect=(.13,0,1,.95));fig.savefig(C_DIR/f'restoration_{kind}.png',dpi=160,bbox_inches='tight')
    plt.show();plt.close(fig)
c_restoration_grid('dropout',0.50)
c_restoration_grid('square',0.40)

### III.7 — Missing-pixel MSE (9 points)

Define the reconstruction error on pixels that were actually missing:

$$\mathrm{MSE}_{\mathrm{missing}} = \frac{\sum_{b,i,j}(1-m_{b,i,j})(\hat x_{b,i,j}-x_{b,i,j})^2}{\max\left(1,\sum_{b,i,j}(1-m_{b,i,j})\right)}.$$

Implement `c_masked_mse(completed, original, mask) -> scalar` for three `[B,1,28,28]` tensors. The two models must be evaluated on the **same masks**. The curves show mean MSE and standard error across three independent corruptions, with severity on the horizontal axis.


In [ ]:
def c_masked_mse(completed, original, mask):
    """Return scalar MSE over missing pixels; if none missing, return zero."""
    # TODO III.7 (9 points): weight errors by 1-mask, normalize accordingly.
    raise NotImplementedError('TODO III.7')


In [ ]:
a=torch.zeros(1,1,2,2)
b=torch.tensor([[[[1.,0.],[1.,0.]]]])
msk=torch.tensor([[[[0.,1.],[0.,1.]]]])
assert torch.allclose(c_masked_mse(a,b,msk),torch.tensor(1.0))
assert torch.allclose(c_masked_mse(a,b,torch.ones_like(msk)),torch.tensor(0.0))
print('PASS — masked-pixel MSE tests')

@torch.no_grad()
def c_compare_mse(clean,kind,severity,repeats=3):
    results={name:[] for name in c_models}
    for _ in range(repeats):
        observed,mask=c_corrupt(clean,kind,severity)
        for name,net in c_models.items():
            completed=c_restore(net,observed,mask)
            results[name].append(float(c_masked_mse(completed,clean,mask)))
    return {name:(float(np.mean(vals)),float(np.std(vals,ddof=1)/np.sqrt(len(vals))) if len(vals)>1 else 0.)
            for name,vals in results.items()}

c_evaluation_batch=c_test_x[:min(512,len(c_test_x))].to(C_DEVICE)
c_scores={}
for c_kind,c_levels in [('dropout',[.20,.35,.50,.65]),('square',[.20,.30,.40,.50])]:
    c_scores[c_kind]={p:c_compare_mse(c_evaluation_batch,c_kind,p,3) for p in c_levels}
fig,axs=plt.subplots(1,2,figsize=(11,3.6),constrained_layout=True)
for ax,(kind,levels) in zip(axs,c_scores.items()):
    x=list(levels)
    for name in c_models:
        means=[levels[p][name][0] for p in x]
        errors=[levels[p][name][1] for p in x]
        ax.errorbar(x,means,yerr=errors,marker='o',capsize=3,label=name)
    ax.set(xlabel=('Fraction of pixels removed' if kind=='dropout' else 'Occlusion side / image width'),
           ylabel='MSE on missing pixels (lower is better)',
           title=('Random pixel dropout' if kind=='dropout' else 'Contiguous square occlusion'))
    ax.grid(alpha=.2);ax.legend(fontsize=8)
fig.suptitle('Missing-pixel reconstruction error under matched corruptions',fontsize=11)
fig.savefig(C_DIR/'missing_pixel_mse.png',dpi=160,bbox_inches='tight')
plt.show();plt.close(fig)
for kind,picks in [('dropout',0.50),('square',0.40)]:
    res=c_scores[kind][picks]
    print(f'{kind} severity {picks:.2f}: '+ ' | '.join(f'{name}: {val[0]:.4f} ± {val[1]:.4f}' for name,val in res.items()))


### III.8 — Results (9 points)

Use your own results and figures to answer the following questions.

1. Which model has a lower missing-pixel MSE for 50% random masking and 40%-width square occlusion? Report the results for both models in each case. (3 points)

2. How does the pattern of missing pixels affect reconstruction? Give one example from your restoration figures. (3 points)

3. Which model reconstructs clean test images better? Compare their clean-input BCE values. What does this difference suggest? (3 points)

Write your answers below.


### References

- D. P. Kingma and M. Welling, [*Auto-Encoding Variational Bayes*](https://arxiv.org/abs/1312.6114), 2014.
- H. Xiao, K. Rasul, and R. Vollgraf, [*Fashion-MNIST: a Novel Image Dataset for Benchmarking Machine Learning Algorithms*](https://arxiv.org/abs/1708.07747), 2017.

